### Node Features Range Distribution

In [1]:
import pandas as pd
from rdkit import Chem
from collections import defaultdict

# Load dataset
df = pd.read_csv('C:/Users/suman/OneDrive/Bureau/Internship_Study/GNN_On_OdorPrediction/data/OdorSmiles_Updated.csv', encoding='ISO-8859-1')
smiles_list = df['SMILES'].dropna().tolist()

# Initialize feature distributions
feature_distribution = {
    'atomic_num': defaultdict(int),
    'degree': defaultdict(int),
    'formal_charge': defaultdict(int),
    'num_hs': defaultdict(int),
    'num_radical_electrons': defaultdict(int),
    'valence': defaultdict(int),
    'smallest_ring': defaultdict(int),
}

for smiles in smiles_list:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        continue

    for atom in mol.GetAtoms():
        feature_distribution['atomic_num'][atom.GetAtomicNum()] += 1
        feature_distribution['degree'][atom.GetDegree()] += 1
        feature_distribution['formal_charge'][atom.GetFormalCharge()] += 1
        feature_distribution['num_hs'][atom.GetTotalNumHs()] += 1
        feature_distribution['num_radical_electrons'][atom.GetNumRadicalElectrons()] += 1
        feature_distribution['valence'][atom.GetTotalValence()] += 1
        feature_distribution['smallest_ring'][atom.GetOwningMol().GetRingInfo().NumAtomRings(atom.GetIdx())] += 1

# Summary
summary = {}
for key, dist in feature_distribution.items():
    summary[key] = {
        "min": min(dist.keys()) if dist else None,
        "max": max(dist.keys()) if dist else None,
        "most_common": max(dist.items(), key=lambda x: x[1]) if dist else None,
        "total_unique_values": len(dist)
    }

# Print results
import pprint
pprint.pprint(summary)


[10:29:35] WARNING: not removing hydrogen atom without neighbors
[10:29:35] WARNING: not removing hydrogen atom without neighbors


{'atomic_num': {'max': 35,
                'min': 1,
                'most_common': (6, 38450),
                'total_unique_values': 13},
 'degree': {'max': 4,
            'min': 0,
            'most_common': (2, 23554),
            'total_unique_values': 5},
 'formal_charge': {'max': 2,
                   'min': -2,
                   'most_common': (0, 45845),
                   'total_unique_values': 5},
 'num_hs': {'max': 4,
            'min': 0,
            'most_common': (0, 12737),
            'total_unique_values': 5},
 'num_radical_electrons': {'max': 1,
                           'min': 0,
                           'most_common': (0, 45921),
                           'total_unique_values': 2},
 'smallest_ring': {'max': 15,
                   'min': 0,
                   'most_common': (0, 29330),
                   'total_unique_values': 9},
 'valence': {'max': 6,
             'min': 0,
             'most_common': (4, 38471),
             'total_unique_values': 7}}


### Molecular Feature Distribution

In [ ]:
# Initialize dictionary to hold molecular features
feature_dict = {
    'molecular_weight': [],
    'logp': [],
    'tpsa': [],
    'num_rings': [],
    'num_rotatable_bonds': [],
    'num_H_bond_donors': [],
    'num_H_bond_acceptors': [],
    'heavy_atom_count': [],
    'formal_charge': [],
    'complexity': []
}

# Compute molecular features for each SMILES
for smiles in df['SMILES']:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        continue

    feature_dict['molecular_weight'].append(Descriptors.MolWt(mol))
    feature_dict['logp'].append(Descriptors.MolLogP(mol))
    feature_dict['tpsa'].append(rdMolDescriptors.CalcTPSA(mol))
    feature_dict['num_rings'].append(rdMolDescriptors.CalcNumRings(mol))
    feature_dict['num_rotatable_bonds'].append(Descriptors.NumRotatableBonds(mol))
    feature_dict['num_H_bond_donors'].append(rdMolDescriptors.CalcNumHBD(mol))
    feature_dict['num_H_bond_acceptors'].append(rdMolDescriptors.CalcNumHBA(mol))
    feature_dict['heavy_atom_count'].append(Descriptors.HeavyAtomCount(mol))
    feature_dict['formal_charge'].append(Chem.GetFormalCharge(mol))
    feature_dict['complexity'].append(Descriptors.FractionCSP3(mol))  # Using CSP3 as a proxy for complexity

# Analyze and print feature distributions
print("\n=== Molecular Feature Distributions ===\n")
for key, values in feature_dict.items():
    series = pd.Series(values)
    print(f"Feature: {key}")
    print(f"  Min: {series.min()}")
    print(f"  Max: {series.max()}")
    print(f"  Mean: {series.mean():.2f}")
    print(f"  Unique Values: {series.nunique()}")
    print(f"  Most Common Values:\n{series.value_counts().head(5)}")
    print("-" * 60)


[10:39:11] WARNING: not removing hydrogen atom without neighbors
[10:39:11] WARNING: not removing hydrogen atom without neighbors



=== Molecular Feature Distributions ===

Feature: molecular_weight
  Min: 17.031
  Max: 6179.372999999956
  Mean: 176.49
  Unique Values: 1333
  Most Common Values:
154.253    47
152.237    44
170.252    39
142.198    30
198.306    29
Name: count, dtype: int64
------------------------------------------------------------
Feature: logp
  Min: -83.66679999999921
  Max: 12.452099999999982
  Mean: 2.51
  Unique Values: 2280
  Most Common Values:
1.9058    21
2.6860    18
2.7119    18
3.4662    15
2.1777    15
Name: count, dtype: int64
------------------------------------------------------------
Feature: tpsa
  Min: 0.0
  Max: 3038.9300000000026
  Mean: 27.21
  Unique Values: 191
  Most Common Values:
26.30    962
17.07    580
20.23    434
0.00     234
18.46    144
Name: count, dtype: int64
------------------------------------------------------------
Feature: num_rings
  Min: 0
  Max: 38
  Mean: 0.86
  Unique Values: 11
  Most Common Values:
1    1632
0    1431
2     449
3     145
4      27